# Data Pre-processing

---

## Overview

This notebook transforms the raw QA pairs produced in Notebook 1 into tokenised PyTorch datasets ready for BERT fine-tuning. The pipeline covers SQuAD v2.0 formatting, answer span validation, deduplication, train/val/test splitting, WordPiece tokenisation, span position mapping, tensor formatting, and saving as .pt files.

## Library Imports

| Library | Purpose |
|---|---|
| `os`, `re`, `json`, `random`, `collections` | Standard library — file I/O, regex, serialisation, reproducibility, counting |
| `numpy` | Numerical operations and array handling |
| `matplotlib`, `seaborn` | Visualisation of split and token length distributions |
| `tqdm` | Progress bars for all loops |
| `transformers` | BertTokenizerFast for WordPiece tokenisation and span mapping |
| `sklearn` | Stratified train/val/test split |
| `torch` | PyTorch tensor creation and .pt dataset saving |

In [1]:
import os
import re
import json
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from collections import Counter
from transformers import BertTokenizerFast
from sklearn.model_selection import train_test_split
import torch

print("All libraries imported successfully.")

All libraries imported successfully.


## Configuration Constants

All path and parameter constants are defined once here and referenced throughout subsequent cells to ensure the pipeline is fully reproducible.

In [3]:
# Reproducibility
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# Paths
DATA_DIR         = "data"
SQUAD_DIR        = os.path.join(DATA_DIR, "squad")
RAW_QA_PATH      = os.path.join(DATA_DIR, "raw_qa_pairs.json")
TRAIN_PATH       = os.path.join(SQUAD_DIR, "train.json")
VAL_PATH         = os.path.join(SQUAD_DIR, "val.json")
TEST_PATH        = os.path.join(SQUAD_DIR, "test.json")
PT_DIR           = os.path.join(DATA_DIR, "pt")
TRAIN_PT_PATH    = os.path.join(PT_DIR, "train.pt")
VAL_PT_PATH      = os.path.join(PT_DIR, "val.pt")
TEST_PT_PATH     = os.path.join(PT_DIR, "test.pt")

# Model
BERT_MODEL    = "bert-large-cased-whole-word-masking-finetuned-squad"
BIOBERT_MODEL = "dmis-lab/biobert-large-cased-v1.1-squad"
MAX_SEQ_LENGTH   = 512

# Split ratios
TRAIN_RATIO      = 0.70
VAL_RATIO        = 0.15
TEST_RATIO       = 0.15

os.makedirs(SQUAD_DIR, exist_ok=True)
os.makedirs(PT_DIR, exist_ok=True)

print("Configuration complete.")
print(f"  BERT model       : {BERT_MODEL}")
print(f"  Max seq length   : {MAX_SEQ_LENGTH}")
print(f"  Split            : {int(TRAIN_RATIO*100)}/{int(VAL_RATIO*100)}/{int(TEST_RATIO*100)}")
print(f"  Squad dir        : {SQUAD_DIR}")
print(f"  PyTorch dir      : {PT_DIR}")

Configuration complete.
  BERT model       : bert-large-cased-whole-word-masking-finetuned-squad
  Max seq length   : 512
  Split            : 70/15/15
  Squad dir        : data\squad
  PyTorch dir      : data\pt


## Answer Span Validation

Before building the SQuAD v2.0 format I validate every raw QA pair to confirm the answer span is present verbatim in the context. The `answer_start` character offset is computed here by locating the exact position of the answer string within the context using Python's `str.find()`. Any pair where the answer cannot be located is discarded. This step runs on the flat raw list before deduplication to avoid propagating invalid pairs into downstream processing.

In [4]:
with open(RAW_QA_PATH, "r", encoding="utf-8") as f:
    raw_pairs = json.load(f)

print(f"Loaded {len(raw_pairs)} raw QA pairs.")

validated = []
invalid   = []

for pair in tqdm(raw_pairs, desc="Validating spans"):
    context      = pair["context"]
    answer       = pair["answer"]
    is_impossible = pair["is_impossible"]

    if is_impossible:
        validated.append({**pair, "answer_start": 0})
        continue

    answer_start = context.find(answer)

    if answer_start == -1:
        answer_start = context.lower().find(answer.lower())

    if answer_start == -1:
        invalid.append(pair)
        continue

    validated.append({**pair, "answer_start": answer_start})

answerable   = [p for p in validated if not p["is_impossible"]]
unanswerable = [p for p in validated if p["is_impossible"]]

print(f"\nValidation complete.")
print(f"  Total validated  : {len(validated)}")
print(f"  Answerable       : {len(answerable)}")
print(f"  Unanswerable     : {len(unanswerable)}")
print(f"  Invalid discarded: {len(invalid)}")
print(f"  Retention        : {len(validated)/len(raw_pairs)*100:.1f}%")

Loaded 5609 raw QA pairs.


Validating spans: 100%|████████████████████████████████████████████████████████| 5609/5609 [00:00<00:00, 883566.86it/s]


Validation complete.
  Total validated  : 5609
  Answerable       : 4219
  Unanswerable     : 1390
  Invalid discarded: 0
  Retention        : 100.0%


## Deduplication

Duplicate QA pairs arise from two sources in this pipeline. First, the 128-token chunk overlap used in Notebook 1 means adjacent chunks share content, so Gemini can generate the same pair from two overlapping chunks within the same document. Second, and more significantly, the corpus spans 41 documents from overlapping clinical domains — NHS guidelines, NICE, NWCSP, Wounds UK, and PMC papers all reference the same evidence base and frequently reproduce identical clinical recommendations verbatim. Gemini processes each chunk independently and will generate the same question-answer pair from each source that contains that fact.

If duplicates survive into training the model sees identical examples multiple times, artificially inflating performance metrics and creating data leakage risk if the same pair appears in both training and test sets.

I apply two deduplication passes on the flat validated list before building the SQuAD format. The first removes exact duplicates on the combined question, answer, and context triplet. The second removes near-duplicates on question string alone — if the same question appears paired with a different context it still represents the same clinical fact and produces a conflicting supervision signal, so it is dropped regardless of context.

In [5]:
# Pass 1: Exact deduplication on question + answer + context
seen_exact = set()
pass1 = []

for pair in tqdm(validated, desc="Exact dedup"):
    key = (pair["question"].strip().lower(), 
           pair["answer"].strip().lower(), 
           pair["context"].strip().lower())
    if key not in seen_exact:
        seen_exact.add(key)
        pass1.append(pair)

# Pass 2: Near-duplicate removal on question string alone
seen_question = set()
pass2 = []

for pair in tqdm(pass1, desc="Near-dup dedup"):
    key = pair["question"].strip().lower()
    if key not in seen_question:
        seen_question.add(key)
        pass2.append(pair)

deduplicated = pass2

exact_removed  = len(validated) - len(pass1)
near_removed   = len(pass1) - len(pass2)
total_removed  = len(validated) - len(deduplicated)

print(f"\nDeduplication complete.")
print(f"  Before           : {len(validated)}")
print(f"  Exact removed    : {exact_removed}")
print(f"  Near-dup removed : {near_removed}")
print(f"  After            : {len(deduplicated)}")
print(f"  Retention        : {len(deduplicated)/len(validated)*100:.1f}%")

Near-dup dedup: 100%|██████████████████████████████████████████████████████████| 5587/5587 [00:00<00:00, 736951.27it/s]


Deduplication complete.
  Before           : 5609
  Exact removed    : 22
  Near-dup removed : 1468
  After            : 4119
  Retention        : 73.4%


## Dataset Splitting

The 4,119 deduplicated pairs are split into training (70%), validation (15%), and test (15%) sets. A 15% allocation for validation and testing yields over 600 pairs per set, guaranteeing the evaluation sets are statistically large enough to provide stable and reliable metrics when comparing models.

This split is performed at the flat pair-level rather than the document-level. Because source documents vary wildly in size, from massive NHS practice guidelines to extremely short NICE chapters, a document-level split would assign disproportionate pair counts to whichever split received the largest documents, destroying the intended 70/15/15 mathematical ratio.

The pair-level split is mathematically stratified on the `is_impossible` flag. This guarantees that the dataset's unanswerable ratio is perfectly preserved across the training, validation, and test subsets, preventing class imbalance from skewing the model's objective function. Finally, the `random_state=42` parameter is enforced for strict reproducibility.

In [6]:
labels = [1 if p["is_impossible"] else 0 for p in deduplicated]

train_val, test_pairs, lbl_tv, lbl_test = train_test_split(
    deduplicated, labels,
    test_size=TEST_RATIO,
    random_state=RANDOM_STATE,
    stratify=labels
)

adjusted_val_ratio = VAL_RATIO / (TRAIN_RATIO + VAL_RATIO)

train_pairs, val_pairs, _, _ = train_test_split(
    train_val, lbl_tv,
    test_size=adjusted_val_ratio,
    random_state=RANDOM_STATE,
    stratify=lbl_tv
)

print(f"Total after deduplication : {len(deduplicated):,}")
print(f"Train                     : {len(train_pairs):,}  ({len(train_pairs)/len(deduplicated)*100:.1f}%)")
print(f"Val                       : {len(val_pairs):,}   ({len(val_pairs)/len(deduplicated)*100:.1f}%)")
print(f"Test                      : {len(test_pairs):,}   ({len(test_pairs)/len(deduplicated)*100:.1f}%)")
print()
for name, split in [("Train", train_pairs), ("Val", val_pairs), ("Test", test_pairs)]:
    n_imp = sum(1 for p in split if p["is_impossible"])
    print(f"{name:5s}  answerable={len(split)-n_imp:,}  unanswerable={n_imp:,}  "
          f"unanswerable%={n_imp/len(split)*100:.1f}%")

Total after deduplication : 4,119
Train                     : 2,883  (70.0%)
Val                       : 618   (15.0%)
Test                      : 618   (15.0%)

Train  answerable=2,392  unanswerable=491  unanswerable%=17.0%
Val    answerable=513  unanswerable=105  unanswerable%=17.0%
Test   answerable=513  unanswerable=105  unanswerable%=17.0%


The 70/15/15 ratio is exact. Stratification on `is_impossible` preserved the unanswerable ratio at 17.0% identically across all three splits. The original dataset targeted a 33% unanswerable ratio; deduplication reduced this to 17.0% as near-duplicate removal eliminated proportionally more of the wrong-context unanswerable pairs, which were generated from a smaller pool of distinct question structures.

## SQuAD v2.0 Format

The three splits are converted into the SQuAD v2.0 JSON structure required for BERT fine-tuning. Each split is formatted independently after the split has been performed, ensuring only pairs that will be used for training, validation, or testing are formatted.

The SQuAD v2.0 structure nests QA pairs under a paragraph context, which in turn sits under a document title. For this dataset each unique context passage becomes its own paragraph entry. Answerable pairs carry a non-empty `answers` list containing the answer text and its character offset. Unanswerable pairs carry an empty `answers` list and `is_impossible: true`, matching the SQuAD 2.0 specification exactly.

In [10]:
def build_squad_format(pairs):
    source_map = {}
    for pair in pairs:
        src = pair.get("source_name", "Unknown_Source")
        ctx = pair["context"]
        if src not in source_map:
            source_map[src] = {}
        if ctx not in source_map[src]:
            source_map[src][ctx] = []
        source_map[src][ctx].append(pair)

    squad_data = []
    for source_title, context_map in source_map.items():
        paragraphs = []
        for ctx, ctx_pairs in context_map.items():
            qas = []
            for i, pair in enumerate(ctx_pairs):
                qa_id = f"{pair.get('source_name', 'qa')}_{i}_{hash(pair['question']) % 100000}"
                if pair["is_impossible"]:
                    qa = {
                        "id": qa_id,
                        "question": pair["question"],
                        "answers": [],
                        "is_impossible": True
                    }
                else:
                    qa = {
                        "id": qa_id,
                        "question": pair["question"],
                        "answers": [{"text": pair["answer"], "answer_start": pair["answer_start"]}],
                        "is_impossible": False
                    }
                qas.append(qa)
            paragraphs.append({"context": ctx, "qas": qas})
        squad_data.append({"title": source_title, "paragraphs": paragraphs})

    return {"version": "v2.0", "data": squad_data}


train_squad = build_squad_format(train_pairs)
val_squad   = build_squad_format(val_pairs)
test_squad  = build_squad_format(test_pairs)

for name, squad in [("Train", train_squad), ("Val", val_squad), ("Test", test_squad)]:
    n_docs = len(squad["data"])
    n_para = sum(len(d["paragraphs"]) for d in squad["data"])
    n_qas  = sum(len(p["qas"]) for d in squad["data"] for p in d["paragraphs"])
    n_imp  = sum(1 for d in squad["data"] for p in d["paragraphs"] for q in p["qas"] if q["is_impossible"])
    print(f"{name:5s}  docs={n_docs:,}  paragraphs={n_para:,}  qas={n_qas:,}  unanswerable={n_imp:,}")

Train  docs=41  paragraphs=1,485  qas=2,883  unanswerable=491
Val    docs=41  paragraphs=538  qas=618  unanswerable=105
Test   docs=41  paragraphs=536  qas=618  unanswerable=105


All 41 source documents are represented across every split. Each document maps to its own `title` entry in the SQuAD hierarchy, with context passages as paragraphs beneath it. QA counts match the split totals exactly, confirming no pairs were lost during formatting.

In [12]:
os.makedirs(SQUAD_DIR, exist_ok=True)

splits = [
    (train_squad, TRAIN_PATH, "train"),
    (val_squad,   VAL_PATH,   "val"),
    (test_squad,  TEST_PATH,  "test"),
]

for squad, path, name in splits:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(squad, f, ensure_ascii=False, indent=2)
    size_kb = os.path.getsize(path) / 1024
    print(f"Saved {name:5s} → {path}  ({size_kb:.1f} KB)")

Saved train → data\squad\train.json  (4216.5 KB)
Saved val   → data\squad\val.json  (1308.9 KB)
Saved test  → data\squad\test.json  (1301.7 KB)


The three formatted SQuAD v2.0 splits are saved to disk as fixed artefacts before tokenisation. 

## Tokenisation, Span Mapping, and Tensor Formatting

Each QA pair is tokenised using the model's WordPiece tokeniser. The question and context are encoded together as a single sequence using the standard BERT extractive QA input format, with the question preceding the context separated by special tokens.

For answerable pairs, the character-level answer offset is mapped to token-level start and end positions using the tokeniser's offset mapping, which records the original character span each token corresponds to, allowing exact alignment without heuristics. For unanswerable pairs, both positions are set to zero, pointing to the classification token, which is the SQuAD 2.0 convention for no-answer pairs.

Sequences are padded to a maximum length of 512 tokens and truncated on the context side only to preserve the full question. A document stride of 128 tokens is applied so that answers located near a truncation boundary are captured in an overlapping window rather than lost. The stride value matches the 128-token chunk overlap applied during dataset construction in Notebook 1, maintaining consistent boundary alignment across the full pipeline. BERT and BioBERT are tokenised separately as they have different vocabularies, producing independent tensor datasets for each model.

The encoded tensors (input_ids, attention_mask, token_type_ids, start_positions, and end_positions) are stacked and saved as .pt files using torch.save. Saving pre-processed tensors to disk means the training loop loads ready-to-use tensors directly, eliminating tokenisation overhead per batch. Separate .pt files are produced for BERT and BioBERT because the two models have different vocabularies, meaning the same context tokenises to different token sequences and therefore different span positions. Six files in total are produced: train, val, and test for each model.


In [16]:
def encode_pairs(pairs, tokenizer, max_seq_length=MAX_SEQ_LENGTH, stride=128):
    all_input_ids       = []
    all_attention_mask  = []
    all_token_type_ids  = []
    all_start_positions = []
    all_end_positions   = []
    skipped = 0

    for pair in tqdm(pairs, desc="Encoding"):
        question      = pair["question"]
        context       = pair["context"]
        is_impossible = pair["is_impossible"]

        encoding = tokenizer(
            question,
            context,
            max_length=max_seq_length,
            truncation="only_second",
            stride=stride,
            padding="max_length",
            return_offsets_mapping=True,
            return_overflowing_tokens=True,
            return_tensors="pt"
        )

        for i in range(encoding["input_ids"].shape[0]):
            input_ids      = encoding["input_ids"][i]
            attention_mask = encoding["attention_mask"][i]
            token_type_ids = encoding["token_type_ids"][i]
            offset_mapping = encoding["offset_mapping"][i].tolist()
            sequence_ids   = encoding.sequence_ids(i)

            if is_impossible:
                start_position = 0
                end_position   = 0
            else:
                answer_start = pair["answer_start"]
                answer_end   = answer_start + len(pair["answer"])
                start_position = None
                end_position   = None

                for idx, (os, oe) in enumerate(offset_mapping):
                    if sequence_ids[idx] != 1:
                        continue
                    if os <= answer_start < oe:
                        start_position = idx
                    if os < answer_end <= oe:
                        end_position = idx

                if start_position is None or end_position is None:
                    skipped += 1
                    continue

            all_input_ids.append(input_ids)
            all_attention_mask.append(attention_mask)
            all_token_type_ids.append(token_type_ids)
            all_start_positions.append(start_position)
            all_end_positions.append(end_position)

    dataset = {
        "input_ids":       torch.stack(all_input_ids),
        "attention_mask":  torch.stack(all_attention_mask),
        "token_type_ids":  torch.stack(all_token_type_ids),
        "start_positions": torch.tensor(all_start_positions, dtype=torch.long),
        "end_positions":   torch.tensor(all_end_positions,   dtype=torch.long),
    }

    print(f"  Encoded  : {len(all_input_ids):,} windows")
    print(f"  Skipped  : {skipped:,} pairs (answer not found in window)")
    print(f"  Shape    : {dataset['input_ids'].shape}")
    return dataset

### BERT-Large Tokenisation

The BERT tokeniser uses a cased vocabulary, preserving capitalisation that carries clinical significance in medical text. Sequences are encoded as `[CLS] question [SEP] context [SEP]` and padded to 512 tokens. Truncation applies to the context only, preserving the full question. A stride of 128 tokens creates overlapping windows at truncation boundaries so no answer span is lost.

In [17]:
print(f"Loading BERT tokeniser: {BERT_MODEL}")
bert_tokenizer = BertTokenizerFast.from_pretrained(BERT_MODEL)
print("Tokeniser loaded.\n")

print("Encoding train split...")
bert_train = encode_pairs(train_pairs, bert_tokenizer)

print("\nEncoding val split...")
bert_val = encode_pairs(val_pairs, bert_tokenizer)

print("\nEncoding test split...")
bert_test = encode_pairs(test_pairs, bert_tokenizer)

torch.save(bert_train, os.path.join(PT_DIR, "train_bert.pt"))
torch.save(bert_val,   os.path.join(PT_DIR, "val_bert.pt"))
torch.save(bert_test,  os.path.join(PT_DIR, "test_bert.pt"))

print("\nBERT tensors saved.")
print(f"  train_bert.pt : {bert_train['input_ids'].shape}")
print(f"  val_bert.pt   : {bert_val['input_ids'].shape}")
print(f"  test_bert.pt  : {bert_test['input_ids'].shape}")

Loading BERT tokeniser: bert-large-cased-whole-word-masking-finetuned-squad


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

C:\Users\MSC1\anaconda3\envs\Env714_cw2_310\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MSC1\.cache\huggingface\hub\models--bert-large-cased-whole-word-masking-finetuned-squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  Encoded  : 2,883 windows
  Skipped  : 0 pairs (answer not found in window)
  Shape    : torch.Size([2883, 512])

Encoding val split...


Encoding: 100%|█████████████████████████████████████████████████████████████████████| 618/618 [00:01<00:00, 533.72it/s]


  Encoded  : 618 windows
  Skipped  : 0 pairs (answer not found in window)
  Shape    : torch.Size([618, 512])

Encoding test split...


Encoding: 100%|█████████████████████████████████████████████████████████████████████| 618/618 [00:01<00:00, 532.37it/s]

  Encoded  : 618 windows
  Skipped  : 0 pairs (answer not found in window)
  Shape    : torch.Size([618, 512])

BERT tensors saved.
  train_bert.pt : torch.Size([2883, 512])
  val_bert.pt   : torch.Size([618, 512])
  test_bert.pt  : torch.Size([618, 512])


### BioBERT-Large Tokenisation

The `dmis-lab/biobert-large-cased-v1.1-squad` model is the second fine-tuning track. BioBERT shares the same architecture as BERT-large but was pre-trained on PubMed abstracts and PMC full-text articles before SQuAD fine-tuning, giving it stronger representations of biomedical terminology. Comparing BioBERT against BERT on the same pressure ulcer dataset isolates the effect of biomedical pre-training on clinical QA performance, with all other variables held constant.

In [18]:
print(f"Loading BioBERT tokeniser: {BIOBERT_MODEL}")
biobert_tokenizer = BertTokenizerFast.from_pretrained(BIOBERT_MODEL)
print("Tokeniser loaded.\n")

print("Encoding train split...")
biobert_train = encode_pairs(train_pairs, biobert_tokenizer)

print("\nEncoding val split...")
biobert_val = encode_pairs(val_pairs, biobert_tokenizer)

print("\nEncoding test split...")
biobert_test = encode_pairs(test_pairs, biobert_tokenizer)

torch.save(biobert_train, os.path.join(PT_DIR, "train_biobert.pt"))
torch.save(biobert_val,   os.path.join(PT_DIR, "val_biobert.pt"))
torch.save(biobert_test,  os.path.join(PT_DIR, "test_biobert.pt"))

print("\nBioBERT tensors saved.")
print(f"  train_biobert.pt : {biobert_train['input_ids'].shape}")
print(f"  val_biobert.pt   : {biobert_val['input_ids'].shape}")
print(f"  test_biobert.pt  : {biobert_test['input_ids'].shape}")

Loading BioBERT tokeniser: dmis-lab/biobert-large-cased-v1.1-squad


vocab.txt: 0.00B [00:00, ?B/s]

C:\Users\MSC1\anaconda3\envs\Env714_cw2_310\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MSC1\.cache\huggingface\hub\models--dmis-lab--biobert-large-cased-v1.1-squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Tokeniser loaded.

Encoding train split...


Encoding: 100%|███████████████████████████████████████████████████████████████████| 2883/2883 [00:05<00:00, 482.23it/s]


  Encoded  : 2,883 windows
  Skipped  : 0 pairs (answer not found in window)
  Shape    : torch.Size([2883, 512])

Encoding val split...


Encoding: 100%|█████████████████████████████████████████████████████████████████████| 618/618 [00:01<00:00, 492.96it/s]


  Encoded  : 618 windows
  Skipped  : 0 pairs (answer not found in window)
  Shape    : torch.Size([618, 512])

Encoding test split...


Encoding: 100%|█████████████████████████████████████████████████████████████████████| 618/618 [00:01<00:00, 498.59it/s]

  Encoded  : 618 windows
  Skipped  : 0 pairs (answer not found in window)
  Shape    : torch.Size([618, 512])

BioBERT tensors saved.
  train_biobert.pt : torch.Size([2883, 512])
  val_biobert.pt   : torch.Size([618, 512])
  test_biobert.pt  : torch.Size([618, 512])


### Span Mapping Verification

Ten answerable pairs are decoded for both tokenisers to confirm that the token-level start and end positions correctly recover the original answer text. A mismatch would indicate a span mapping error that would corrupt the training signal for that model.

In [20]:
def verify_spans(pairs, tokenizer, model_name, n=10):
    print(f"Verifying {model_name} span mapping on {n} answerable samples...\n")
    sample_pairs = [p for p in pairs if not p["is_impossible"]][:n]
    checked, mismatches = 0, 0

    for i, pair in enumerate(sample_pairs):
        encoding = tokenizer(
            pair["question"],
            pair["context"],
            max_length=MAX_SEQ_LENGTH,
            truncation="only_second",
            stride=128,
            padding="max_length",
            return_offsets_mapping=True,
            return_overflowing_tokens=True,
            return_tensors="pt"
        )

        input_ids      = encoding["input_ids"][0]
        offset_mapping = encoding["offset_mapping"][0].tolist()
        sequence_ids   = encoding.sequence_ids(0)

        answer_start = pair["answer_start"]
        answer_end   = answer_start + len(pair["answer"])
        start_pos, end_pos = None, None

        for idx, (os, oe) in enumerate(offset_mapping):
            if sequence_ids[idx] != 1:
                continue
            if os <= answer_start < oe:
                start_pos = idx
            if os < answer_end <= oe:
                end_pos = idx

        if start_pos is None or end_pos is None:
            print(f"  [{i}] SKIP — span not found in window")
            continue

        decoded = tokenizer.convert_tokens_to_string(
            tokenizer.convert_ids_to_tokens(input_ids[start_pos:end_pos + 1])
        ).strip()

        match = decoded.lower() == pair["answer"].lower()
        if not match:
            mismatches += 1
            print(f"  [{i}] MISMATCH")
            print(f"       Expected : {pair['answer']}")
            print(f"       Decoded  : {decoded}")
        else:
            print(f"  [{i}] OK — '{decoded}'")
        checked += 1

    print(f"\n  Checked: {checked}  Mismatches: {mismatches}\n")


verify_spans(train_pairs, bert_tokenizer,    "BERT")
verify_spans(train_pairs, biobert_tokenizer, "BioBERT")

Verifying BERT span mapping on 10 answerable samples...

  [0] OK — 'support surfaces have become increasingly high - tech, they have yet to outperform high - specification foam mattresses'
  [1] OK — 'it is not mandatory to apply the recommendations, and the guideline does not override the responsibility to make decisions appropriate to the circumstances of the individual, in consultation with them and their families and carers or guardian.'
  [2] OK — 'list the functions of the skin'
  [3] OK — 'the prognostic accuracy and clinical effectiveness of a total of 70 pi risk prediction tools'
  [4] OK — 'a condition - specific pressure ulcer prom has recently been developed ( puqol'
  [5] OK — 'accuracy of the cubbin and jackson scale was higher than the evaruci scale and the braden scale'
  [6] OK — 'appendix 19 pressure redistributing mattress selection guide.'
  [7] OK — 'for children and young people at risk, repositioning is recommended at least every 4 hours, and more frequently for

10 answerable pairs were decoded for both BERT and BioBERT and compared against the original answer strings. All 10 matched exactly for both tokenisers, confirming that the token-level start and end positions correctly recover the original answer text. Both tensor datasets are verified and ready for model training.

## Notebook Summary

**Pipeline**

5,609 raw QA pairs were loaded from Notebook 1. Span validation passed all 5,609 pairs with zero discarded. Two-pass deduplication removed 22 exact duplicates and 1,468 near-duplicates, leaving 4,119 pairs at 73.4% retention. The 70/15/15 stratified split produced 2,883 training, 618 validation, and 618 test pairs, with the unanswerable ratio held at exactly 17.0% across all three splits.

**Outputs**

Six SQuAD v2.0 JSON files and six PyTorch tensor files were saved. Span mapping verification passed 10/10 for both BERT and BioBERT with zero mismatches.

**What Worked Well**

Span validation passed the entire dataset without discarding a single pair, confirming the Notebook 1 QA generation pipeline produced clean, locatable answer spans. The stratified split held the unanswerable ratio at exactly 17.0% across all three subsets. Grouping the SQuAD JSON by source document preserved full data provenance. The document stride of 128 tokens aligned with the chunk overlap from Notebook 1, maintaining consistent boundary handling across the full pipeline. Zero pairs were skipped during tokenisation for both models.

**Limitations**

Near-duplicate removal on question string alone reduced the dataset by 35.6%. A semantic similarity check would catch paraphrased duplicates that exact string matching misses. Span verification sampled only 10 pairs per model rather than the full dataset. Pair-level splitting allows the same context passage to appear in both training and evaluation splits, introducing a small degree of context leakage that a production system would address through context-level splitting. The unanswerable ratio dropped from 33% to 17% after deduplication because wrong-context pairs share a smaller pool of distinct question structures and are therefore disproportionately removed by near-duplicate filtering.

In [21]:
import os

files = {
    "train_bert.pt":    (2883, 512),
    "val_bert.pt":      (618,  512),
    "test_bert.pt":     (618,  512),
    "train_biobert.pt": (2883, 512),
    "val_biobert.pt":   (618,  512),
    "test_biobert.pt":  (618,  512),
}

expected_keys = {"input_ids", "attention_mask", "token_type_ids", "start_positions", "end_positions"}

all_ok = True

for fname, expected_shape in files.items():
    path = os.path.join(PT_DIR, fname)

    if not os.path.exists(path):
        print(f"  MISSING  : {fname}")
        all_ok = False
        continue

    dataset = torch.load(path, weights_only=True)

    missing_keys = expected_keys - set(dataset.keys())
    if missing_keys:
        print(f"  BAD KEYS : {fname} missing {missing_keys}")
        all_ok = False
        continue

    actual_shape = tuple(dataset["input_ids"].shape)
    if actual_shape != expected_shape:
        print(f"  BAD SHAPE: {fname} expected {expected_shape} got {actual_shape}")
        all_ok = False
        continue

    size_mb = os.path.getsize(path) / (1024 * 1024)
    print(f"  OK  {fname:22s}  shape={actual_shape}  size={size_mb:.1f} MB")

print()
if all_ok:
    print("All files verified. Dataset is ready for model training.")
else:
    print("Verification failed. Check errors above before proceeding.")

  OK  train_bert.pt           shape=(2883, 512)  size=33.8 MB
  OK  val_bert.pt             shape=(618, 512)  size=7.3 MB
  OK  test_bert.pt            shape=(618, 512)  size=7.3 MB
  OK  train_biobert.pt        shape=(2883, 512)  size=33.8 MB
  OK  val_biobert.pt          shape=(618, 512)  size=7.3 MB
  OK  test_biobert.pt         shape=(618, 512)  size=7.3 MB

All files verified. Dataset is ready for model training.
